In [ ]:
import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# run utils.db_setup to set up duckdb database
conn = duckdb.connect('event.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

## Schema

<img src="https://relbench.stanford.edu/img/rel-event.png" alt="schema" width="800"/>

## Tasks

### `user-attendance`

Regression -- MAE

In [ ]:
%%sql train_df <<
from user_attendance_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.user.nunique():,} users.')
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / train_df.user.nunique()
)
print()
print('Label Distribution:')
mean = train_df.target.mean()
std = train_df.target.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(train_df.target == 0).mean():.1%}')
print(f'Quantiles: \n{train_df.target.quantile(np.arange(0, 1.1, 0.1))}')

In [ ]:
%%sql val_df <<
from user_attendance_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.user.nunique():,} users.')
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / val_df.user.nunique()
)
print()
print(f'Users in val seen in train: {len(set(val_df.user) & set(train_df.user)):,}')
print()
print('Label Distribution:')
mean = val_df.target.mean()
std = val_df.target.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(val_df.target == 0).mean():.1%}')
print(f'Quantiles: \n{val_df.target.quantile(np.arange(0, 1.1, 0.1))}')

### `user-ignore`

Binary Classification -- ROCAUC

In [ ]:
%%sql train_df <<
select * from user_ignore_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.user.nunique():,} users.')
print()
print('Label Distribution:')
print(train_df.target.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / train_df.user.nunique()
)

In [ ]:
%%sql val_df <<
select * from user_ignore_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.user.nunique():,} users.')
print()
print('Label Distribution:')
print(val_df.target.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'Users in val seen in train: {len(set(val_df.user) & set(train_df.user)):,}')
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / val_df.user.nunique()
)

### `user-repeat`

In [ ]:
%%sql train_df <<
select * from user_repeat_train;

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.user.nunique():,} users.')
print()
print('Label Distribution:')
print(train_df.target.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(train_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / train_df.user.nunique()
)

In [ ]:
%%sql val_df <<
select * from user_repeat_val;

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.user.nunique():,} users.')
print()
print('Label Distribution:')
print(val_df.target.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(val_df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'Users in val seen in train: {len(set(val_df.user) & set(train_df.user)):,}')
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='user', keep='last')
    .groupby('timestamp').user.nunique() / val_df.user.nunique()
)

## Users

In [ ]:
%%sql users <<
select * from users;

In [ ]:
users

In [ ]:
users.locale.value_counts().head(10)

In [ ]:
users.location.isna().value_counts()

In [ ]:
users.timezone.value_counts(normalize=True).head(10)

In [ ]:
users.gender.value_counts(normalize=True)

In [ ]:
fig = px.histogram(users, x='birthyear', color='gender', range_x=(1940, 2005), opacity=0.7, barmode='overlay')
fig.update_layout(width=700, height=500, xaxis_title='Birth Year', yaxis_title='Count')
fig.show(renderer='png')

## User Friends

In [ ]:
%%sql user_friends <<
select * from user_friends;

In [ ]:
user_friends.friend.isna().mean()

In [ ]:
user_friends

In [ ]:
user_friends.friend.isna().mean()

In [ ]:
from relbench.datasets import get_dataset

In [ ]:
ds = get_dataset('rel-event', download=True)

In [ ]:
ds.get_db().table_dict['user_friends']

## Events

In [ ]:
%%sql events <<
select * from events;

In [ ]:
events

According to docs, `c_i` columns represent the number of times the ith most common word stem appears in the name or description of the event.

In [ ]:
events.user_id.isna().mean()

In [ ]:
events.country.value_counts(normalize=True, dropna=False).head(10)

In [ ]:
events.state.value_counts(normalize=True, dropna=False).head(10)

In [ ]:
fig = px.histogram(events.start_time.dt.strftime('%Y-%m'), title='Event Start Time Distribution', range_x=['2011-12', '2013-01'])
fig.update_layout(width=700, height=500, xaxis_title='Birth Year', yaxis_title='Count')
fig.show(renderer='png')

In [ ]:
events.start_time.describe()

## Event Attendees

In [ ]:
%%sql attendees <<
select * from event_attendees;

In [ ]:
attendees

In [ ]:
(
    attendees
    .groupby(['event', 'status'])
    .start_time
    .count()
    .reset_index()
    .groupby('status')
    .start_time
    .agg(['mean', 'std', 'min', 'max'])
)

In [ ]:
invites = attendees.groupby('event').user_id.count()
fig = px.histogram(
    x=(invites + 1), title='Invites per Event', histnorm='percent', labels={'x': 'Num Invites'},
    log_x=True, log_y=True
)
fig.update_layout(width=700, height=500)
fig.show(renderer='png')

## Event Interest

In [ ]:
%%sql interest <<
select * from event_interest;

In [ ]:
interest

In [ ]:
interest.groupby('event')[['invited', 'interested', 'not_interested']].sum().describe()

In [ ]:
interest.groupby('user')[['invited', 'interested', 'not_interested']].sum().describe()